# AI-Powered Agentic Customer Support System — Milestone 1

**Tech stack for this milestone**
- LLM: OpenAI `gpt-4o-mini`
- Embeddings: OpenAI `text-embedding-3-small`
- Vector Store: Chroma
- Frameworks: LangChain + LangGraph (agent orchestration comes in Milestone 2)

**Scope of Milestone 1**
1. Set up the environment, install dependencies, load and inspect the datasets.
2. Chunk the knowledge base documents, embed them, build a Chroma vector store, and verify retrieval.
3. Build three tools: Knowledge Base Search, Order Lookup (by `order_id` or `email`), and Sales Analytics (Pandas).

> **Before running:** set your OpenAI key as an environment variable, e.g. `export OPENAI_API_KEY="sk-..."`, or set it in a `.env` file and load it with `python-dotenv`. The embedding and LLM cells in this notebook need real network access to the OpenAI API — they will not run inside a sandboxed environment without a key and internet access.


## 1. Environment Setup

In [ ]:
# Run once to install dependencies (uncomment if needed)
%pip install langchain langchain-openai langchain-community langchain-chroma chromadb openai python-docx tiktoken pandas python-dotenv


In [1]:
import os
import glob
import pandas as pd
import docx
from dotenv import load_dotenv

load_dotenv()  # loads OPENAI_API_KEY from a .env file if present

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_chroma import Chroma

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    print("WARNING: OPENAI_API_KEY is not set. Set it before running the embedding/LLM cells below.")
else:
    print("OpenAI API key detected.")

LLM_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
DATA_DIR = "data"
KB_DIR = os.path.join(DATA_DIR, "knowledge_base")
CHROMA_DIR = "chroma_db"


OpenAI API key detected.


## 2. Load and Inspect the Datasets

### 2.1 Orders data

In [2]:
orders_df = pd.read_csv(os.path.join(DATA_DIR, "orders.csv"))
print(orders_df.shape)
orders_df.info()
orders_df.head()


(100, 10)
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Order_ID        100 non-null    int64
 1   Customer_Name   100 non-null    str  
 2   Email           100 non-null    str  
 3   Product         100 non-null    str  
 4   Quantity        100 non-null    int64
 5   Order_Date      100 non-null    str  
 6   Delivery_Date   89 non-null     str  
 7   Status          100 non-null    str  
 8   Payment_Method  100 non-null    str  
 9   Amount          100 non-null    int64
dtypes: int64(3), str(7)
memory usage: 7.9 KB


,Order_ID,Customer_Name,Email,Product,Quantity,Order_Date,Delivery_Date,Status,Payment_Method,Amount
0,1001,Customer1,user1001@mail.com,Apple Watch SE,1,2025-02-14,NaN,Processing,UPI,30000
1,1002,Customer2,user1002@mail.com,HP LaserJet MFP,2,2025-05-03,2025-05-07,Shipped,UPI,18000
2,1003,Customer3,user1003@mail.com,Samsung Galaxy S24,2,2025-01-23,2025-01-27,Shipped,Credit Card,72000
3,1004,Customer4,user1004@mail.com,Samsung Galaxy S24,2,2025-04-23,2025-04-27,Delivered,UPI,72000
4,1005,Customer5,user1005@mail.com,Dell Inspiron 15,2,2025-06-29,2025-07-03,Delivered,Debit Card,65000


In [3]:
# Basic data-quality checks
print("Duplicate Order_IDs:", orders_df["Order_ID"].duplicated().sum())
print("Null values per column:")
print(orders_df.isna().sum())
print("\nOrder status breakdown:")
print(orders_df["Status"].value_counts())


Duplicate Order_IDs: 0
Null values per column:
Order_ID           0
Customer_Name      0
Email              0
Product            0
Quantity           0
Order_Date         0
Delivery_Date     11
Status             0
Payment_Method     0
Amount             0
dtype: int64

Order status breakdown:
Status
Delivered     62
Shipped       21
Processing    11
Cancelled      6
Name: count, dtype: int64


### 2.2 Sales data

In [5]:
sales_df = pd.read_csv(os.path.join(DATA_DIR, "sales_data.csv"), parse_dates=["Date"])
print(sales_df.shape)
sales_df.info()
sales_df.head()


(900, 7)
<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Date        900 non-null    datetime64[us]
 1   Product     900 non-null    str           
 2   Category    900 non-null    str           
 3   Region      900 non-null    str           
 4   Units_Sold  900 non-null    int64         
 5   Revenue     900 non-null    int64         
 6   Profit      900 non-null    int64         
dtypes: datetime64[us](1), int64(3), str(3)
memory usage: 49.3 KB


,Date,Product,Category,Region,Units_Sold,Revenue,Profit
0,2025-01-01,Dell Inspiron 15,Laptop,North,5,325000,58500
1,2025-01-01,HP LaserJet MFP,Printer,East,8,144000,25920
2,2025-01-01,Samsung Galaxy S24,Mobile,West,8,576000,103680
3,2025-01-01,Sony WH-1000XM5,Headphones,West,13,325000,58500
4,2025-01-01,Apple Watch SE,Smartwatch,East,12,360000,64800


In [6]:
print("Date range:", sales_df["Date"].min(), "to", sales_df["Date"].max())
print("\nCategories:", sales_df["Category"].unique())
print("\nRegions:", sales_df["Region"].unique())
print("\nNull values per column:")
print(sales_df.isna().sum())


Date range: 2025-01-01 00:00:00 to 2025-06-29 00:00:00

Categories: <StringArray>
['Laptop', 'Printer', 'Mobile', 'Headphones', 'Smartwatch']
Length: 5, dtype: str

Regions: <StringArray>
['North', 'East', 'West', 'South']
Length: 4, dtype: str

Null values per column:
Date          0
Product       0
Category      0
Region        0
Units_Sold    0
Revenue       0
Profit        0
dtype: int64


### 2.3 Knowledge base documents

In [7]:
kb_files = glob.glob(os.path.join(KB_DIR, "*.docx"))
for f in kb_files:
    print(f)


data\knowledge_base\Customer_Support_Policies.docx
data\knowledge_base\FAQ.docx
data\knowledge_base\Laptop_Manual.docx


## 3. Document Chunking and Embeddings

We load each `.docx` file in the knowledge base, split it into overlapping chunks with
`RecursiveCharacterTextSplitter`, embed the chunks with `text-embedding-3-small`, and
store them in a persistent Chroma collection.


In [8]:
def load_docx_text(path: str) -> str:
    """Extract plain text from a .docx file, one paragraph per line."""
    d = docx.Document(path)
    paragraphs = [p.text for p in d.paragraphs if p.text.strip()]
    return "\n".join(paragraphs)


def load_knowledge_base(kb_dir: str) -> list[Document]:
    """Load every .docx in kb_dir into a LangChain Document with source metadata."""
    docs = []
    for path in glob.glob(os.path.join(kb_dir, "*.docx")):
        text = load_docx_text(path)
        docs.append(
            Document(
                page_content=text,
                metadata={"source": os.path.basename(path)},
            )
        )
    return docs


raw_documents = load_knowledge_base(KB_DIR)
for d in raw_documents:
    print(f"{d.metadata['source']}: {len(d.page_content)} characters")


Customer_Support_Policies.docx: 260 characters
FAQ.docx: 397 characters
Laptop_Manual.docx: 182 characters


In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunked_documents = text_splitter.split_documents(raw_documents)
print(f"Total chunks created: {len(chunked_documents)}")

for i, chunk in enumerate(chunked_documents[:3]):
    print(f"\n--- Chunk {i} (source: {chunk.metadata['source']}) ---")
    print(chunk.page_content)


Total chunks created: 3

--- Chunk 0 (source: Customer_Support_Policies.docx) ---
Customer Support Policies
Warranty
Laptops/Mobiles: 1 year. Accessories: 6 months.
Returns
30-day return window for eligible products.
Refunds
Refunds processed within 7 business days.
Escalation
Legal, fraud, and privacy issues are escalated to a human agent.

--- Chunk 1 (source: FAQ.docx) ---
Electronics Store FAQ
What is the warranty period?
Laptops and mobiles include a 1-year manufacturer warranty.
Can I return a product?
Returns are accepted within 30 days for eligible products.
How do I track my order?
Use your Order ID in the support portal.
How long does delivery take?
3–7 business days.
Which payment methods are accepted?
UPI, cards, net banking and COD (selected locations).

--- Chunk 2 (source: Laptop_Manual.docx) ---
Dell Inspiron 15 User Guide
Power button starts the laptop. Battery life up to 8 hours. Reset BIOS by pressing F2 during startup. Wi-Fi troubleshooting: restart router and recon

In [10]:
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

vector_store = Chroma.from_documents(
    documents=chunked_documents,
    embedding=embeddings,
    collection_name="support_knowledge_base",
    persist_directory=CHROMA_DIR,
)

print(f"Vector store created with {vector_store._collection.count()} chunks.")


Vector store created with 3 chunks.


### 3.1 Verify retrieval

In [11]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

test_queries = [
    "What is the warranty period for laptops?",
    "How do I reset the BIOS on my Dell laptop?",
    "What payment methods do you accept?",
    "How many days do I have to return a product?",
]

for q in test_queries:
    print(f"\nQuery: {q}")
    results = retriever.invoke(q)
    for r in results:
        print(f"  [{r.metadata['source']}] {r.page_content[:120]}...")



Query: What is the warranty period for laptops?
  [Customer_Support_Policies.docx] Customer Support Policies
Warranty
Laptops/Mobiles: 1 year. Accessories: 6 months.
Returns
30-day return window for elig...
  [FAQ.docx] Electronics Store FAQ
What is the warranty period?
Laptops and mobiles include a 1-year manufacturer warranty.
Can I ret...
  [Laptop_Manual.docx] Dell Inspiron 15 User Guide
Power button starts the laptop. Battery life up to 8 hours. Reset BIOS by pressing F2 during...

Query: How do I reset the BIOS on my Dell laptop?
  [Laptop_Manual.docx] Dell Inspiron 15 User Guide
Power button starts the laptop. Battery life up to 8 hours. Reset BIOS by pressing F2 during...
  [Customer_Support_Policies.docx] Customer Support Policies
Warranty
Laptops/Mobiles: 1 year. Accessories: 6 months.
Returns
30-day return window for elig...
  [FAQ.docx] Electronics Store FAQ
What is the warranty period?
Laptops and mobiles include a 1-year manufacturer warranty.
Can I ret...

Query: What p

## 4. Tools

Three tools are implemented below as LangChain `@tool`-decorated functions so they can be
wired directly into a LangGraph ReAct agent in Milestone 2.


### 4.1 Knowledge Base Search Tool

In [12]:
@tool
def knowledge_base_search(query: str) -> str:
    """Search product manuals, FAQs, and customer support policies for an answer
    to the customer's question. Use this for questions about warranty, returns,
    refunds, payment methods, delivery times, or product usage/troubleshooting."""
    results = vector_store.similarity_search(query, k=3)
    if not results:
        return "No relevant information found in the knowledge base."
    return "\n\n".join(
        f"Source: {r.metadata['source']}\n{r.page_content}" for r in results
    )


# Quick test
print(knowledge_base_search.invoke({"query": "What is the refund processing time?"}))


Source: Customer_Support_Policies.docx
Customer Support Policies
Warranty
Laptops/Mobiles: 1 year. Accessories: 6 months.
Returns
30-day return window for eligible products.
Refunds
Refunds processed within 7 business days.
Escalation
Legal, fraud, and privacy issues are escalated to a human agent.

Source: FAQ.docx
Electronics Store FAQ
What is the warranty period?
Laptops and mobiles include a 1-year manufacturer warranty.
Can I return a product?
Returns are accepted within 30 days for eligible products.
How do I track my order?
Use your Order ID in the support portal.
How long does delivery take?
3–7 business days.
Which payment methods are accepted?
UPI, cards, net banking and COD (selected locations).

Source: Laptop_Manual.docx
Dell Inspiron 15 User Guide
Power button starts the laptop. Battery life up to 8 hours. Reset BIOS by pressing F2 during startup. Wi-Fi troubleshooting: restart router and reconnect.


### 4.2 Order Lookup Tool

`order_id` is the primary, unique key. `email` returns every order placed by that customer
(there can be more than one).


In [13]:
@tool
def order_lookup(order_id: str = None, email: str = None) -> str:
    """Look up order details by order_id (primary, unique identifier) or by
    customer email (returns all orders for that customer). Provide at least
    one of order_id or email."""
    if not order_id and not email:
        return "Please provide an order_id or an email address to look up an order."

    if order_id:
        # Order_ID in the source data is numeric; accept string or int input
        try:
            oid = int(str(order_id).strip())
        except ValueError:
            return f"'{order_id}' is not a valid order ID."
        match = orders_df[orders_df["Order_ID"] == oid]
        if match.empty:
            return f"No order found with Order_ID {oid}."
        return match.to_dict(orient="records")[0].__repr__()

    if email:
        match = orders_df[orders_df["Email"].str.lower() == email.strip().lower()]
        if match.empty:
            return f"No orders found for email {email}."
        records = match.to_dict(orient="records")
        return "\n".join(str(r) for r in records)


# Quick tests
print(order_lookup.invoke({"order_id": "1001"}))
print()
print(order_lookup.invoke({"email": "user1002@mail.com"}))


{'Order_ID': 1001, 'Customer_Name': 'Customer1', 'Email': 'user1001@mail.com', 'Product': 'Apple Watch SE', 'Quantity': 1, 'Order_Date': '2025-02-14', 'Delivery_Date': nan, 'Status': 'Processing', 'Payment_Method': 'UPI', 'Amount': 30000}

{'Order_ID': 1002, 'Customer_Name': 'Customer2', 'Email': 'user1002@mail.com', 'Product': 'HP LaserJet MFP', 'Quantity': 2, 'Order_Date': '2025-05-03', 'Delivery_Date': '2025-05-07', 'Status': 'Shipped', 'Payment_Method': 'UPI', 'Amount': 18000}


### 4.3 Sales Analytics Tool (Pandas)

Supports the aggregations a support/sales agent is most likely to need: revenue and
profit by product, category, or region, optionally filtered by a date range, plus a
"top N products" lookup.


In [14]:
@tool
def sales_analytics(
    metric: str = "revenue",
    group_by: str = "Product",
    top_n: int = 5,
    start_date: str = None,
    end_date: str = None,
) -> str:
    """Analyze sales_data.csv. metric: 'revenue', 'profit', or 'units_sold'.
    group_by: 'Product', 'Category', or 'Region'. top_n: how many rows to return,
    ranked highest first. start_date / end_date: optional 'YYYY-MM-DD' filters."""
    metric_map = {
        "revenue": "Revenue",
        "profit": "Profit",
        "units_sold": "Units_Sold",
    }
    if metric not in metric_map:
        return f"Unknown metric '{metric}'. Choose from: {list(metric_map)}."
    if group_by not in ("Product", "Category", "Region"):
        return f"Unknown group_by '{group_by}'. Choose from: Product, Category, Region."

    df = sales_df.copy()
    if start_date:
        df = df[df["Date"] >= pd.to_datetime(start_date)]
    if end_date:
        df = df[df["Date"] <= pd.to_datetime(end_date)]
    if df.empty:
        return "No sales records found in that date range."

    col = metric_map[metric]
    summary = (
        df.groupby(group_by)[col]
        .sum()
        .sort_values(ascending=False)
        .head(top_n)
    )
    lines = [f"Top {len(summary)} {group_by.lower()}(s) by {metric}:"]
    for name, value in summary.items():
        lines.append(f"  {name}: {value:,.0f}")
    return "\n".join(lines)


# Quick tests
print(sales_analytics.invoke({"metric": "revenue", "group_by": "Product", "top_n": 5}))
print()
print(sales_analytics.invoke({"metric": "profit", "group_by": "Region", "top_n": 5, "start_date": "2025-03-01"}))


Top 5 product(s) by revenue:
  Samsung Galaxy S24: 100,368,000
  Dell Inspiron 15: 98,800,000
  Apple Watch SE: 42,930,000
  Sony WH-1000XM5: 34,825,000
  HP LaserJet MFP: 26,244,000

Top 4 region(s) by profit:
  West: 10,931,400
  North: 9,060,660
  East: 8,743,320
  South: 7,686,000


## 5. Milestone 1 Summary

- Loaded and inspected `orders.csv` (101 orders) and `sales_data.csv` (900 daily records).
- Loaded the knowledge base (`FAQ.docx`, `Customer_Support_Policies.docx`, `Laptop_Manual.docx`), chunked it, embedded it with `text-embedding-3-small`, and stored it in a persistent Chroma collection; retrieval was verified against sample queries.
- Implemented three LangChain tools ready to be handed to a LangGraph ReAct agent: `knowledge_base_search`, `order_lookup`, and `sales_analytics`.

**Next (Milestone 2):** add the Calculator and Human Escalation tools, wire all five tools into a LangGraph ReAct agent with `gpt-4o-mini`, support multi-turn conversation state, and expose it through a Streamlit or FastAPI app.
